# 04 — Multi-Coin 1m Historical Loader

Fetches 30 days of 1-minute OHLCV data for all Roostoo coins from Binance via CCXT.
Stores everything in `../data/ohlcv_1m.duckdb` in long format (one row per candle).

**Note:** Some Roostoo-listed coins may not exist on Binance — these are skipped automatically.

## 0. Imports & Config

In [1]:
import duckdb
import pandas as pd
import ccxt
import time
import os
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

load_dotenv('../.env')

DB_PATH = '../data/ohlcv_1m.duckdb'
os.makedirs('../data', exist_ok=True)
con = duckdb.connect(DB_PATH)
print(f'Connected to {DB_PATH}')

Connected to ../data/ohlcv_1m.duckdb


## 1. Create Table

In [2]:
con.execute("""
    CREATE TABLE IF NOT EXISTS ohlcv (
        ts          TIMESTAMPTZ NOT NULL,
        symbol      VARCHAR     NOT NULL,
        interval    VARCHAR     NOT NULL,
        open        DOUBLE,
        high        DOUBLE,
        low         DOUBLE,
        close       DOUBLE,
        volume      DOUBLE,
        source      VARCHAR DEFAULT 'binance',
        PRIMARY KEY (ts, symbol, interval)
    )
""")
print('Table ready.')

Table ready.


## 2. Define Target Symbols (Alphabetical)

In [3]:
# Roostoo X/USD → Binance X/USDT → DuckDB X-USD (alphabetical)
COINS = [
    '1000CHEEMS',
    'AAVE',
    'ADA',
    'APT',
    'ARB',
    'ASTER',
    'AVAX',
    'AVNT',
    'BIO',
    'BMT',
    'BNB',
    'BONK',
    'BTC',
    'CAKE',
    'CFX',
    'CRV',
    'DOGE',
    'DOT',
    'EDEN',
    'EIGEN',
    'ENA',
    'ETH',
    'FET',
    'FIL',
    'FLOKI',
    'FORM',
    'HBAR',
    'HEMI',
    'ICP',
    'LINK',
    'LINEA',
    'LISTA',
    'LTC',
    'MIRA',
    'NEAR',
    'OMNI',
    'ONDO',
    'OPEN',
    'PAXG',
    'PEPE',
    'PENDLE',
    'PENGU',
    'PLUME',
    'POL',
    'PUMP',
    'S',
    'SEI',
    'SHIB',
    'SOL',
    'SOMI',
    'STO',
    'SUI',
    'TAO',
    'TON',
    'TRUMP',
    'TRX',
    'TUT',
    'UNI',
    'VIRTUAL',
    'WIF',
    'WLD',
    'WLFI',
    'XLM',
    'XPL',
    'XRP',
    'ZEC',
    'ZEN',
]

TARGETS = [
    {'binance': f'{c}/USDT', 'duckdb': f'{c}-USD'}
    for c in COINS
]

INTERVAL   = '1m'
LOOKBACK   = 30
BATCH_SIZE = 1000

since_dt = datetime.now(timezone.utc) - timedelta(days=LOOKBACK)
since_ms = int(since_dt.timestamp() * 1000)

print(f'Coins:    {len(TARGETS)}')
print(f'Interval: {INTERVAL}')
print(f'From:     {since_dt.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Expected: ~{len(TARGETS) * LOOKBACK * 24 * 60:,} total rows')

Coins:    67
Interval: 1m
From:     2026-02-16 07:10 UTC
Expected: ~2,894,400 total rows


## 3. Fetch & Insert All Coins

In [4]:
exchange = ccxt.binance({'enableRateLimit': True})

def fetch_all_ohlcv(exchange, symbol, timeframe, since_ms, batch_size=1000):
    all_candles = []
    current_since = since_ms
    while True:
        candles = exchange.fetch_ohlcv(
            symbol, timeframe=timeframe,
            since=current_since, limit=batch_size
        )
        if not candles:
            break
        all_candles.extend(candles)
        current_since = candles[-1][0] + 1
        if len(candles) < batch_size:
            break
        time.sleep(0.1)
    return all_candles

skipped = []
loaded  = []

for i, target in enumerate(TARGETS):
    binance_sym = target['binance']
    db_sym      = target['duckdb']
    print(f'[{i+1}/{len(TARGETS)}] Fetching {binance_sym}...', end=' ')

    try:
        raw = fetch_all_ohlcv(exchange, binance_sym, INTERVAL, since_ms, BATCH_SIZE)
        if not raw:
            print('No data — skipped.')
            skipped.append(binance_sym)
            continue

        df = pd.DataFrame(raw, columns=['ts_ms', 'open', 'high', 'low', 'close', 'volume'])
        df['ts']       = pd.to_datetime(df['ts_ms'], unit='ms', utc=True)
        df['symbol']   = db_sym
        df['interval'] = INTERVAL
        df['source']   = 'binance'
        df = df[['ts', 'symbol', 'interval', 'open', 'high', 'low', 'close', 'volume', 'source']]
        df = df.drop_duplicates(subset=['ts', 'symbol', 'interval'])

        con.execute("""
            INSERT OR REPLACE INTO ohlcv
            SELECT ts, symbol, interval, open, high, low, close, volume, source
            FROM df
        """)

        print(f'{len(df):,} rows inserted.')
        loaded.append(db_sym)

    except Exception as e:
        print(f'ERROR — skipped. ({e})')
        skipped.append(binance_sym)

print(f'\nDone. Loaded: {len(loaded)} coins | Skipped: {len(skipped)} coins')
if skipped:
    print(f'Skipped: {skipped}')

[1/67] Fetching 1000CHEEMS/USDT... 43,200 rows inserted.
[2/67] Fetching AAVE/USDT... 43,200 rows inserted.
[3/67] Fetching ADA/USDT... 43,200 rows inserted.
[4/67] Fetching APT/USDT... 43,200 rows inserted.
[5/67] Fetching ARB/USDT... 43,201 rows inserted.
[6/67] Fetching ASTER/USDT... 43,201 rows inserted.
[7/67] Fetching AVAX/USDT... 43,201 rows inserted.
[8/67] Fetching AVNT/USDT... 43,201 rows inserted.
[9/67] Fetching BIO/USDT... 43,201 rows inserted.
[10/67] Fetching BMT/USDT... 43,201 rows inserted.
[11/67] Fetching BNB/USDT... 43,202 rows inserted.
[12/67] Fetching BONK/USDT... 43,202 rows inserted.
[13/67] Fetching BTC/USDT... 43,202 rows inserted.
[14/67] Fetching CAKE/USDT... 43,202 rows inserted.
[15/67] Fetching CFX/USDT... 43,202 rows inserted.
[16/67] Fetching CRV/USDT... 43,202 rows inserted.
[17/67] Fetching DOGE/USDT... 43,203 rows inserted.
[18/67] Fetching DOT/USDT... 43,203 rows inserted.
[19/67] Fetching EDEN/USDT... 43,203 rows inserted.
[20/67] Fetching EIGEN/U

## 4. Verify

In [5]:
summary = con.execute("""
    SELECT symbol,
           COUNT(*)   AS rows,
           MIN(ts)    AS first_ts,
           MAX(ts)    AS last_ts
    FROM ohlcv
    GROUP BY symbol
    ORDER BY symbol
""").fetchdf()

print(f'Total rows: {summary["rows"].sum():,}')
print(f'Coins loaded: {len(summary)}')
summary

Total rows: 2,851,560
Coins loaded: 66


,symbol,rows,first_ts,last_ts
0,1000CHEEMS-USD,43200,2026-02-16 15:11:00+08:00,2026-03-18 15:10:00+08:00
1,AAVE-USD,43200,2026-02-16 15:11:00+08:00,2026-03-18 15:10:00+08:00
2,ADA-USD,43200,2026-02-16 15:11:00+08:00,2026-03-18 15:10:00+08:00
3,APT-USD,43200,2026-02-16 15:11:00+08:00,2026-03-18 15:10:00+08:00
4,ARB-USD,43201,2026-02-16 15:11:00+08:00,2026-03-18 15:11:00+08:00
...,...,...,...,...
61,XLM-USD,43210,2026-02-16 15:11:00+08:00,2026-03-18 15:20:00+08:00
62,XPL-USD,43211,2026-02-16 15:11:00+08:00,2026-03-18 15:21:00+08:00
63,XRP-USD,43211,2026-02-16 15:11:00+08:00,2026-03-18 15:21:00+08:00
64,ZEC-USD,43211,2026-02-16 15:11:00+08:00,2026-03-18 15:21:00+08:00


## 5. Close

In [6]:
con.close()
print('Done. Data saved to data/ohlcv_1m.duckdb')

Done. Data saved to data/ohlcv_1m.duckdb
